# Sync RAS Curations

Syncs verb-based RAS curations (add/remove institution_ids) from the users Heroku Postgres database to a local Databricks table.

**Source**: `openalex_users.public.ras_institution_curations` (Heroku Postgres view, migration 067)
**Target**: `openalex.institutions.ras_curations` (Delta table)

Curations use verb-based semantics:
- `action='add'`: Include this institution_id even if model didn't predict it
- `action='remove'`: Exclude this institution_id even if model predicted it

**View contract**: one row per (raw_affiliation_string, institution_id) *pair* carrying the winning action — NOT one row per curation row, unlike the other `_curations` views. Latest-action-wins (the oxjob #582 toggle fix) lives in the view, in the same repo as the write path; rationale in the migration 067 header. This notebook is just the array pivot; the sync is insert/update-only (RAS curations are an append-only event log — undo = the opposite action, and row deletion is blocked at the API).

Timestamps on the target:
- `latest_curation_at` = MAX(source `created`) per string — when a curator last acted (the honest change signal)
- `updated_datetime` = when this sync last ran (rewritten every night)


## Sync curations from users DB


In [ ]:
%sql
-- Preview what will be synced. The view already resolved latest-action-wins
-- per (string, institution) pair (migration 067, oxjob #582/#684); this is
-- purely the scalar -> array pivot (arrays stay on this side — PG arrays
-- over Lakehouse Federation are fragile).
SELECT
  raw_affiliation_string,
  FILTER(
    ARRAY_AGG(CASE WHEN action = 'add' THEN institution_id END),
    x -> x IS NOT NULL
  ) AS curated_add_ids,
  FILTER(
    ARRAY_AGG(CASE WHEN action = 'remove' THEN institution_id END),
    x -> x IS NOT NULL
  ) AS curated_remove_ids,
  MAX(created) AS latest_curation_at,
  COUNT(*) AS num_pairs
FROM openalex_users.public.ras_institution_curations
GROUP BY raw_affiliation_string


In [ ]:
%sql
-- MERGE curations into local table (inserts + updates only — deliberately NO
-- NOT MATCHED BY SOURCE DELETE). The curations log is append-only for RAS:
-- undo = submit the opposite action (latest-action-wins in the source view,
-- migration 067); row deletion is blocked at the API. A row present here but
-- absent from the view would mean an out-of-band PG delete — preserved here
-- rather than silently un-applied.
MERGE INTO openalex.institutions.ras_curations AS target
USING (
  SELECT
    raw_affiliation_string,
    FILTER(
      ARRAY_AGG(CASE WHEN action = 'add' THEN institution_id END),
      x -> x IS NOT NULL
    ) AS curated_add_ids,
    FILTER(
      ARRAY_AGG(CASE WHEN action = 'remove' THEN institution_id END),
      x -> x IS NOT NULL
    ) AS curated_remove_ids,
    MAX(created) AS latest_curation_at,
    CURRENT_TIMESTAMP() AS updated_datetime
  FROM openalex_users.public.ras_institution_curations
  GROUP BY raw_affiliation_string
) AS source
ON target.raw_affiliation_string = source.raw_affiliation_string
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *


## Verify sync results


In [ ]:
%sql
-- Check local curations table
SELECT 
  COUNT(*) AS total_curated_ras,
  SUM(SIZE(curated_add_ids)) AS total_adds,
  SUM(SIZE(curated_remove_ids)) AS total_removes,
  MAX(latest_curation_at) AS last_curator_action,
  MAX(updated_datetime) AS last_sync
FROM openalex.institutions.ras_curations


In [ ]:
%sql
-- Sample of recently curated RAS
SELECT * FROM openalex.institutions.ras_curations
ORDER BY latest_curation_at DESC
LIMIT 10
